# Exploratory Data Analysis — Medical Insurance Cost Prediction

This notebook explores the insurance dataset through visualisations and statistical analysis.

**Goal:** Understand the relationships between policyholder attributes and insurance charges to inform feature engineering and model selection.

---

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path for imports
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils import load_raw_data, TARGET_COLUMN
from src.preprocessing import clean_dataframe
from src.feature_engineering import engineer_features

# Style
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["figure.figsize"] = (10, 6)

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

print("✅ Imports loaded successfully")

---
## 1. Load Data

Load the raw dataset, clean it, and engineer features.

In [ ]:
# Load raw data
raw = load_raw_data()
print(f"Raw dataset shape: {raw.shape}")
print(f"Columns: {list(raw.columns)}")
print(f"\nFirst 5 rows:")
raw.head()

In [ ]:
# Clean the dataframe
cleaned = clean_dataframe(raw)
print(f"Cleaned dataset shape: {cleaned.shape}")
print(f"Duplicate removed: {len(raw) - len(cleaned)} row(s)")
print(f"Missing values: {cleaned.isna().sum().sum()}")

In [ ]:
# Engineer features
enriched = engineer_features(cleaned)
new_cols = [c for c in enriched.columns if c not in cleaned.columns]
print(f"Enriched dataset shape: {enriched.shape}")
print(f"New engineered features ({len(new_cols)}): {new_cols}")
enriched.head()

---
## 2. Basic Statistics

Descriptive statistics for numeric and categorical features.

In [ ]:
# Numeric feature summary
num_cols = cleaned.select_dtypes(include=[np.number]).columns.tolist()
cleaned[num_cols].describe().T.round(2)

In [ ]:
# Categorical feature distributions
cat_cols = ["sex", "smoker", "region"]
for col in cat_cols:
    print(f"\n{col.upper()} distribution:")
    print(cleaned[col].value_counts())
    print(f"  → {cleaned[col].value_counts(normalize=True).mul(100).round(1).to_dict()} (%)")

---
## 3. Target Variable Analysis

Deep dive into the `charges` distribution — the prediction target.

In [ ]:
# Target statistics
charges = cleaned[TARGET_COLUMN]
print(f"Min:     ${charges.min():,.2f}")
print(f"Max:     ${charges.max():,.2f}")
print(f"Mean:    ${charges.mean():,.2f}")
print(f"Median:  ${charges.median():,.2f}")
print(f"Std Dev: ${charges.std():,.2f}")
print(f"Skewness: {charges.skew():.3f}  ← right-skewed (positive)")
print(f"Kurtosis: {charges.kurtosis():.3f}")

# Quartiles
q1, q3 = charges.quantile(0.25), charges.quantile(0.75)
iqr = q3 - q1
print(f"\nQ1:  ${q1:,.2f}")
print(f"Q3:  ${q3:,.2f}")
print(f"IQR: ${iqr:,.2f}")

# Outliers (1.5×IQR rule)
n_outliers = ((charges < q1 - 1.5*iqr) | (charges > q3 + 1.5*iqr)).sum()
print(f"\nPotential outliers (1.5×IQR): {n_outliers} ({n_outliers/len(charges)*100:.1f}%)")

In [ ]:
# Charges distribution histogram + KDE
fig, ax = plt.subplots(figsize=(12, 5))
sns.histplot(charges, bins=50, kde=True, ax=ax, color="steelblue")
ax.axvline(charges.mean(),   color="red",    lw=2, ls="--", label=f"Mean ${charges.mean():,.0f}")
ax.axvline(charges.median(), color="orange", lw=2, ls="--", label=f"Median ${charges.median():,.0f}")
ax.set(title="Insurance Charges Distribution", xlabel="Charges (USD)", ylabel="Count")
ax.legend()
plt.tight_layout()
plt.show()

---
## 4. Smoker Status — The Strongest Predictor

Smokers incur dramatically higher insurance charges. This is the single most important feature.

In [ ]:
# Smoker vs non-smoker charges
smoker_grp = cleaned.groupby("smoker")[TARGET_COLUMN].agg(["mean", "median", "std", "count"])
smoker_grp.index = ["Non-Smoker", "Smoker"]
smoker_grp["mean"]   = smoker_grp["mean"].apply(lambda x: f"${x:,.0f}")
smoker_grp["median"] = smoker_grp["median"].apply(lambda x: f"${x:,.0f}")
smoker_grp["std"]    = smoker_grp["std"].apply(lambda x: f"${x:,.0f}")
print(smoker_grp)

# Ratio
smoker_mean = cleaned[cleaned["smoker"] == "yes"][TARGET_COLUMN].mean()
nonsmoker_mean = cleaned[cleaned["smoker"] == "no"][TARGET_COLUMN].mean()
print(f"\n🚬 Smokers pay {smoker_mean / nonsmoker_mean:.2f}× more on average.")

In [ ]:
# Boxplot: charges by smoker status
fig, ax = plt.subplots(figsize=(8, 6))
sns.boxplot(data=cleaned, x="smoker", y=TARGET_COLUMN,
            palette={"yes": "#e05c5c", "no": "#5c8ae0"}, ax=ax)
ax.set(title="Insurance Charges by Smoker Status",
       xlabel="Smoker", ylabel="Charges (USD)")
plt.tight_layout()
plt.show()

---
## 5. Age and Charges

Age has a positive, nonlinear relationship with charges. Older individuals pay significantly more, especially smokers.

In [ ]:
# Age vs charges scatter (coloured by smoker status)
fig, ax = plt.subplots(figsize=(12, 6))
for smoker_val, grp in cleaned.groupby("smoker"):
    color = "#e05c5c" if smoker_val == "yes" else "#5c8ae0"
    label = "Smoker" if smoker_val == "yes" else "Non-Smoker"
    ax.scatter(grp["age"], grp[TARGET_COLUMN], alpha=0.5, s=20, color=color, label=label)
ax.set(title="Age vs Insurance Charges", xlabel="Age", ylabel="Charges (USD)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Average charges by age group
age_grp = enriched.groupby("age_group")[TARGET_COLUMN].mean().sort_values(ascending=False)
print("Average charges by age group:")
for grp, val in age_grp.items():
    print(f"  {grp:12s} → ${val:,.0f}")

---
## 6. BMI and Obesity

BMI ≥ 30 (obesity threshold) significantly impacts charges, especially when combined with smoking.

In [ ]:
# BMI vs charges scatter (coloured by smoker status)
fig, ax = plt.subplots(figsize=(12, 6))
for smoker_val, grp in cleaned.groupby("smoker"):
    color = "#e05c5c" if smoker_val == "yes" else "#5c8ae0"
    label = "Smoker" if smoker_val == "yes" else "Non-Smoker"
    ax.scatter(grp["bmi"], grp[TARGET_COLUMN], alpha=0.5, s=20, color=color, label=label)
ax.axvline(30, color="black", ls="--", lw=1.5, label="Obesity threshold (BMI=30)")
ax.set(title="BMI vs Insurance Charges", xlabel="BMI", ylabel="Charges (USD)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Average charges by BMI category
bmi_grp = enriched.groupby("bmi_category")[TARGET_COLUMN].mean()
print("Average charges by BMI category:")
for cat in ["underweight", "normal", "overweight", "obese"]:
    if cat in bmi_grp.index:
        print(f"  {cat:12s} → ${bmi_grp[cat]:,.0f}")

In [ ]:
# Smoker × Obesity interaction — highest-risk group
enriched["group"] = enriched.apply(
    lambda r: "Smoker + Obese" if r["smoker_obese"] == 1
              else "Smoker only" if r["smoker"] == "yes"
              else "Obese only" if r["is_obese"] == 1
              else "Neither",
    axis=1
)
interaction = enriched.groupby("group")[TARGET_COLUMN].mean().sort_values(ascending=False)
print("\nAverage charges by smoker × obesity interaction:")
for grp, val in interaction.items():
    print(f"  {grp:18s} → ${val:,.0f}")
print("\n🔥 Smoker + Obese group pays the most — justifies the `smoker_obese` feature.")

---
## 7. Correlation Analysis

Numeric feature correlations with the target.

In [ ]:
# Correlation heatmap (numeric features only)
num_df = enriched.select_dtypes(include=[np.number])
fig, ax = plt.subplots(figsize=(14, 10))
corr = num_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.5, ax=ax, cbar_kws={"shrink": 0.8})
ax.set_title("Correlation Heatmap — Numeric Features", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Top correlations with charges
corr_with_target = num_df.corr()[TARGET_COLUMN].drop(TARGET_COLUMN).sort_values(ascending=False)
print("Correlation with charges (sorted):")
for feat, val in corr_with_target.items():
    print(f"  {feat:20s}  {val:+.4f}")

---
## 8. Region and Other Categorical Features

In [ ]:
# Average charges by region
region_grp = cleaned.groupby("region")[TARGET_COLUMN].mean().sort_values(ascending=False)
print("Average charges by region:")
for region, val in region_grp.items():
    print(f"  {region:12s} → ${val:,.0f}")

# Variation is relatively small — region is a weak predictor

In [ ]:
# Average charges by sex
sex_grp = cleaned.groupby("sex")[TARGET_COLUMN].mean().sort_values(ascending=False)
print("\nAverage charges by sex:")
for sex, val in sex_grp.items():
    print(f"  {sex:8s} → ${val:,.0f}")

# Sex has minimal impact — difference is < $1,400

In [ ]:
# Average charges by number of children
children_grp = cleaned.groupby("children")[TARGET_COLUMN].mean()
print("\nAverage charges by number of children:")
for n, val in children_grp.items():
    print(f"  {int(n)} children → ${val:,.0f}")

---
## 9. Pairplot — Key Features

Visualise pairwise relationships between numeric features, coloured by smoker status.

In [ ]:
# Pairplot (sample 500 rows for speed)
sample_cols = ["age", "bmi", "children", TARGET_COLUMN, "smoker"]
sample = cleaned[sample_cols].sample(min(500, len(cleaned)), random_state=42)

g = sns.pairplot(sample, hue="smoker",
                 palette={"yes": "#e05c5c", "no": "#5c8ae0"},
                 diag_kind="kde", plot_kws={"alpha": 0.4, "s": 15})
g.fig.suptitle("Pairplot — Key Features by Smoker Status", y=1.01, fontsize=14)
plt.show()

---
## 10. Engineered Features Impact

Validate the 8 derived features created in `src/feature_engineering.py`.

In [ ]:
# Summary of all engineered features
eng_cols = ["age_group", "bmi_category", "is_obese", "smoker_obese",
            "age_bmi", "has_children", "family_size", "age_smoker"]
print("Engineered features — value counts:\n")
for col in eng_cols:
    print(f"{col}:")
    print(enriched[col].value_counts())
    print()

In [ ]:
# Average charges by engineered categorical features
for col in ["age_group", "bmi_category", "family_size"]:
    grp = enriched.groupby(col)[TARGET_COLUMN].mean().sort_values(ascending=False)
    print(f"\nAverage charges by {col}:")
    for cat, val in grp.items():
        print(f"  {cat:18s} → ${val:,.0f}")

---
## 11. Key Takeaways

1. **Smoker status** is the dominant predictor — smokers pay **3.8× more** on average.
2. **BMI ≥ 30 + smoking** creates the highest-charge group — `smoker_obese` interaction captures this.
3. **Age** has a nonlinear effect — charges spike after 50 → age grouping is useful.
4. **Region and sex** have minimal impact — weak predictors.
5. **Children** show slight variation, but engineered `has_children` binary feature is cleaner.
6. **Charges are right-skewed** — skewness ≈ 1.52 with 139 high-value outliers (real data, not errors).
7. **Feature engineering** successfully creates 8 meaningful features that encode domain knowledge.

---
## Next Steps

- Run `python -m src.train` to train 10 models on this enriched dataset.
- Evaluate with RMSE, MAE, R², and 5-fold cross-validation.
- Deploy the best model via the Next.js dashboard and FastAPI backend.\n
---